In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np

In [18]:
headers = {
    'User-Agent':
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36'
}
webpage=requests.get('https://www.ambitionbox.com/list-of-companies?page=1',headers=headers).text

<p>The website ambitionbox doesnt allow direct fetching of data and thats why the headers dict is created to show that the data is being fetched by an user through a browser. (that can be same for all such types)


The ".text" is used tp convert the webpage content to an actual HTML text.</p>

In [19]:
soup=BeautifulSoup(webpage,'lxml')

<h1>FINDING OUT COMPANY NAMES</h1>

In [23]:
for i in soup.find_all('h2',class_='companyCardWrapper__companyName'):
  print(i.text.strip())

TCS
Accenture
Wipro
Cognizant
Capgemini
HDFC Bank
Infosys
HCLTech
ICICI Bank
Tech Mahindra
Genpact
TP
Jio
Axis Bank
Concentrix Corporation
Amazon
Reliance Retail
iEnergizer
LTIMindtree
IBM


<h1>FINDING OUT RATINGS</h1>

In [24]:
for i in soup.find_all('div',class_='rating_text'):
  print(i.text.strip())

3.3
3.7
3.6
3.7
3.6
3.8
3.5
3.4
4.0
3.4
3.6
3.9
4.4
3.6
3.6
3.9
3.9
4.6
3.6
3.9


<h1>FINDING OUT THE NUMBER OF REVIEWS</h1>

In [27]:
for i in soup.find_all('span',class_='companyCardWrapper__companyRatingCount'):
    print(i.text.strip())

(1.1L)
(73.7k)
(65.3k)
(61.6k)
(53.6k)
(52.6k)
(49k)
(46.1k)
(46.1k)
(43.4k)
(42.5k)
(38.8k)
(33.7k)
(33.4k)
(32.4k)
(31.5k)
(27.4k)
(27.2k)
(26.5k)
(25.8k)


<h1>FINDING OUT THE COMPANY FIELD & LOCATION</h1>

In [37]:
for i in soup.find_all('span',class_='companyCardWrapper__interLinking'):
    print(i.text.strip())

IT Services & Consulting | Bengaluru +444 other locations
IT Services & Consulting | Bengaluru +259 other locations
IT Services & Consulting | Hyderabad +373 other locations
IT Services & Consulting | Hyderabad +232 other locations
IT Services & Consulting | Bengaluru +182 other locations
Banking | Mumbai +1853 other locations
IT Services & Consulting | Bengaluru +246 other locations
IT Services & Consulting | Chennai +231 other locations
Banking | Mumbai +1445 other locations
IT Services & Consulting | Hyderabad +322 other locations
Analytics & KPO | Hyderabad +181 other locations
BPO | Mumbai +256 other locations
Telecom | Mumbai +1957 other locations
Banking | Mumbai +1520 other locations
BPO | Bengaluru +183 other locations
Internet | Bengaluru +515 other locations
Retail | Mumbai +1161 other locations
BPO | Noida +53 other locations
IT Services & Consulting | Bengaluru +145 other locations
IT Services & Consulting | Bengaluru +162 other locations


<p>We scraped all the data individually but we need it together so we will now consider the whole container for every company and then store data accordingly</p>

<h1>CONSIDERING THE WHOLE CONTAINER </h1>

In [38]:
company=soup.find_all('div',class_='companyCardWrapper__primaryInformation')

In [69]:
name=[]
rating=[]
reviews=[]
ctype=[]
loc=[]

for i in company:
    name.append(i.find('h2',class_='companyCardWrapper__companyName').text.strip())
    rating.append(i.find('div',class_='rating_text').text.strip())
    reviews.append(i.find('span',class_='companyCardWrapper__companyRatingCount').text.strip())

    #splitting the company type and the company location

    try:
            text=i.find('span',class_='companyCardWrapper__interLinking').text.strip()
            c,l=[x.strip() for x in text.split('|')]
            
    except ValueError:
            c = text
            l = 'N/A'
            
    ctype.append(c)
    loc.append(l)


df=pd.DataFrame({'name':name,'rating':rating,'reviews':reviews,'ctype':ctype,'loc':loc})

<h2>For only page 1</h2>

In [70]:
df

,name,rating,reviews,ctype,loc
0,Honda Motorcycle & Scooter,4.1,(3.5k),Automobile,Gurugram +301 other locations
1,Vishal Mega Mart,3.7,(3.5k),Retail,New Delhi +379 other locations
2,Hindustan Coca Cola Beverages,4.0,(3.5k),Beverage,Bengaluru +263 other locations
3,UBS,3.8,(3.4k),Financial Services,Pune +19 other locations
4,Barclays,3.7,(3.4k),Banking,Pune +38 other locations
5,Bajaj General Insurance Limited,3.7,(3.4k),Insurance,Pune +306 other locations
6,DTDC Express,3.7,(3.4k),Logistics,New Delhi +273 other locations
7,IndusInd Nippon Life Insurance,3.4,(3.4k),Mumbai +493 other locations,N/A
8,ABB,4.0,(3.4k),Industrial Machinery,Bengaluru +108 other locations
9,Ceat Tyres,3.9,(3.4k),Auto Components,Nagpur +141 other locations


<h1>Creating the DataFrame for all pages (500)</h1>

In [73]:
final=pd.DataFrame()
for j in range(1,501):

    url='https://www.ambitionbox.com/list-of-companies?page={}'.format(j)
    headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36'}
    webpage=requests.get(url,headers=headers).text

    soup=BeautifulSoup(webpage,'lxml')

    company=soup.find_all('div',class_='companyCardWrapper__primaryInformation')

    name=[]
    rating=[]
    reviews=[]
    ctype=[]
    loc=[]
    
    for i in company:
        
        # name
        try:
                name.append(
                    i.find(
                        'h2',
                        class_='companyCardWrapper__companyName'
                    ).text.strip()
                )
    
        except:
            name.append('N/A')
    
    
        # rating
        try:
            rating.append(
                i.find(
                    'div',
                    class_='rating_text'
            ).text.strip()
        )

        except:
            rating.append('N/A')
    
    
        # reviews
        try:
            reviews.append(
                i.find(
                    'span',
                    class_='companyCardWrapper__companyRatingCount'
                ).text.strip()
            )
    
        except:
                reviews.append('N/A')
        
    
        # company type and location
        try:
            text = i.find(
                'span',
                class_='companyCardWrapper__interLinking'
            ).text.strip()
    
            c, l = [x.strip() for x in text.split('|')]
    
        except ValueError:
            c = text
            l = 'N/A'
    
        except:
            c = 'N/A'
            l = 'N/A'
    
    
        ctype.append(c)
        loc.append(l)
    
    
    df=pd.DataFrame({'name':name,'rating':rating,'reviews':reviews,'ctype':ctype,'loc':loc})

    final=pd.concat([final,df],ignore_index=True)

In [74]:
final

,name,rating,reviews,ctype,loc
0,TCS,3.3,(1.1L),IT Services & Consulting,Bengaluru +444 other locations
1,Accenture,3.7,(73.7k),IT Services & Consulting,Bengaluru +259 other locations
2,Wipro,3.6,(65.3k),IT Services & Consulting,Hyderabad +373 other locations
3,Cognizant,3.7,(61.6k),IT Services & Consulting,Hyderabad +232 other locations
4,Capgemini,3.6,(53.6k),IT Services & Consulting,Bengaluru +182 other locations
...,...,...,...,...,...
9995,Institute of Liver & Biliary Sciences,4.0,(110),Education & Training,New Delhi +1 other locations
9996,Flowserve Sanmar,3.5,(110),Chemicals,Chennai +15 other locations
9997,DeliverHealth Solutions,3.5,(110),Software Product,Bengaluru +6 other locations
9998,Sree Behariji Mills,3.3,(110),Consumer Electronics & Appliances,Patna +34 other locations


<h1>Converting the DataFrame to CSV</h1>

In [75]:
final.to_csv('companies.csv')